In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

from setting_for_sda.path_setting import path_list
from setting_for_sda.date_setting import Date_Setting


import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from lib.utils.datetime_handler import calc_rel_period
from lib.visualization.distribution_collector import (calc_top_mid_bottom_tags_prop, compute_proportion_period)
from lib.visualization.plot_generator import PlotGen
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()

import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import spearmanr



Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [ ]:
# ---------------- 지표 1: Baseline dominance decay ----------------
def baseline_dominance(df, k=30, cutoff=0,
                       period_col='rel_week', tag_col='tag', prop_col='proportion'):
    """
    Pre 기간의 Top-K 태그가 각 주차에서 차지하는 총 점유율.
    감소할수록 주류 태그가 밀려나고 있음 = 다양화의 증거.
    """
    pre = df[df[period_col] < cutoff]
    k = int(len(pre[tag_col].unique())*0.2)
    baseline_tags = pre.groupby(tag_col)[prop_col].sum().nlargest(k).index.tolist()
    baseline_set = set(baseline_tags)

    rows = {}
    for week, grp in df.groupby(period_col):
        total = grp[prop_col].sum()
        if total == 0:
            continue
        base_share = grp[grp[tag_col].isin(baseline_set)][prop_col].sum() / total
        rows[week] = base_share

    s = pd.Series(rows, name=f'baseline_top{k}_share').sort_index()
    try:
        s.index = s.index.astype(int)
    except (ValueError, TypeError):
        pass
    return s, baseline_tags


# ---------------- 지표 2: Long-tail share ----------------
def longtail_share(df, k=30,
                   period_col='rel_week', prop_col='proportion'):
    """
    각 주차에서 Top-K 밖의 태그들이 차지하는 점유율.
    증가할수록 롱테일이 두꺼워지고 있음 = 다양화.
    
    주의: Top-K 자체도 N-robust 지표지만, "k 이후"를 보는 건 K에 따라 의미가 바뀜.
    여기선 baseline Top-K를 쓰지 않고, 매 주차마다 그 주차의 Top-K 밖을 본다.
    """
    rows = {}
    for week, grp in df.groupby(period_col):
        sorted_props = grp[prop_col].sort_values(ascending=False)
        total = sorted_props.sum()
        n_tags = len(sorted_props)
        
        if total == 0 or n_tags < 2:
            rows[week] = 0.0
            continue
        
        # 하위 20% 태그 수
        k_bottom = max(1, int(n_tags * 0.2))
        
        # 하위 k_bottom개 태그의 점유율 합
        bottom_share = sorted_props.tail(k_bottom).sum() / total
        rows[week] = bottom_share

    s = pd.Series(rows, name=f'longtail_beyond_top{k}').sort_index()
    try:
        s.index = s.index.astype(int)
    except (ValueError, TypeError):
        pass
    return s


def plot_panelB(series_,
                reference_point=0, reference_label='ChatGPT launch',
                figsize=(12, 6), save_path=None, lang = 'python',smooth_window=4, option = 'baseline'):
    """
    4-panel 그림:
    (A) 질문 수 N 시계열 — 감소 추세 확인
    (B) Baseline Top-30 share — 주류 태그 밀려남
    (C) Long-tail share — 롱테일 부상
    (D) Gini: raw vs N-adjusted — N 통제해도 변화 있음
    """
    fig, axes = plt.subplots(1, 1, figsize=figsize, sharex=True)
    
    def smooth(v, idx):
        if smooth_window and smooth_window > 1:
            return pd.Series(v, index=idx).rolling(
                smooth_window, center=True, min_periods=1).mean().values
        return v

        # (B) Baseline share
    idx = series_.index.values
    vals = series_.values
    axes.plot(idx, vals, marker='o', markersize=2, linewidth=0.8,
                 color='#2E86AB', alpha=0.4, label='raw')
    axes.plot(idx, smooth(vals, idx), linewidth=2.2, color='#2E86AB',
                 label=f'{smooth_window}-week avg')
    axes.set_ylabel(f'{option.capitalize()} Top-30 share', fontsize=11)
    axes.set_title(f'(B) {option.capitalize()} share',
                      fontsize=11, loc='left')
    axes.grid(alpha=0.3)
    axes.legend(loc='best', fontsize=9)

    fig.suptitle('Evidence for topic diversification despite declining volume',
                 fontsize=13, y=0.995, fontweight='bold')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()




In [3]:
for idx, lang in enumerate(CONSTANTS.languages_from2020to2022) :
    plotgen = PlotGen(idx, 'tag')
    print(f'[Start....] drawing figures for {lang} language')


    df = load_df(plotgen.data_dir, ['cdate' , 'id' , 'tag', 'cnt', 'tot_cnt', 'pct'])
    df = calc_rel_period(df, plotgen.std_date, date_col = 'cdate', period = 7)
    df_proportion = compute_proportion_period(df,period = 'rel_week', type = 'tag', value_col='pct')

    # 지표 1: Baseline dominance
    baseline_s, bt = baseline_dominance(df_proportion, k=30, cutoff=0)
    print(f"Baseline top 30 (first 5): {bt[:5]}")
    print(f"Baseline share — Pre: {baseline_s.loc[baseline_s.index<0].mean():.3f}, "
          f"Post: {baseline_s.loc[baseline_s.index>=0].mean():.3f}")

    # 지표 2: Long-tail share
    longtail_s = longtail_share(df_proportion, k=30)
    print(f"Long-tail share — Pre: {longtail_s.loc[longtail_s.index<0].mean():.3f}, "
          f"Post: {longtail_s.loc[longtail_s.index>=0].mean():.3f}")


    # 종합 시각화
    plot_panelB(baseline_s, reference_point=0,
                             save_path=f'./fig/{idx}_PanelB_for_{lang}_baseline.png', lang = lang, option='baseline')
        # 종합 시각화
    plot_panelB(longtail_s, reference_point=0,
                             save_path=f'./fig/{idx}_PanelB_for_{lang}_longtail.png', lang = lang, option='longtail')


    print("\nSaved figures.")

[Start....] drawing figures for python language
Baseline top 30 (first 5): ['pandas', 'python-3.x', 'dataframe', 'django', 'numpy']
Baseline share — Pre: 0.025, Post: 0.056
Long-tail share — Pre: 0.581, Post: 0.667

Saved figures.
[Start....] drawing figures for javascript language
Baseline top 30 (first 5): ['reactjs', 'html', 'node.js', 'jquery', 'css']
Baseline share — Pre: 0.025, Post: 0.058
Long-tail share — Pre: 0.467, Post: 0.520

Saved figures.
[Start....] drawing figures for java language
Baseline top 30 (first 5): ['android', 'spring-boot', 'spring', 'maven', 'arrays']
Baseline share — Pre: 0.044, Post: 0.071
Long-tail share — Pre: 0.633, Post: 0.595

Saved figures.
[Start....] drawing figures for c# language
Baseline top 30 (first 5): ['.net', 'unity-game-engine', 'asp.net-core', 'asp.net', 'wpf']
Baseline share — Pre: 0.041, Post: 0.064
Long-tail share — Pre: 0.565, Post: 0.548

Saved figures.
[Start....] drawing figures for c++ language
Baseline top 30 (first 5): ['templat